# Chapter 14 — Basic Reduction Operations

Part 6: Reductions, scans, and combined recipes.

**Thread — one idea.** A reduction is one combine operator over one axis: the axis disappears, every other axis keeps its size, and masked lanes must carry the operator's neutral element.

**Thread — the chain.** 14.1–14.5 scalar reductions (`sum`, `max`, `min`, `argmax`, `argmin`) → 14.6–14.8 statistics (`mean`, `variance`, `std`) → 14.9–14.12 axis and masked reductions → Method 01 reference → Method 02 kernels → Method 03 fusion.

**Ladder:** naive Python/PyTorch reductions (Method 01) → Triton 1-D and 2-D reduction kernels (Method 02) → fused moments and masked normalization statistics (Method 03).

**Study tip.** Fix the $2\times3$ matrix $X=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$ and write two trees — one for axis $0$, one for axis $1$ — plus each output shape with and without keepdim. If you can fill the worked-example table by hand, every kernel is just that table in registers.

**Learning objectives**

- Read a reduction as a tree that combines $N$ lanes into one, and state the output shape from the axis and keepdim rules.
- Use `tl.sum`, `tl.max`, `tl.min`, `tl.argmax`, and `tl.argmin` with the correct axis and neutral element.
- Compute mean, variance, and standard deviation in one fused pass, and explain the cancellation risk in the one-pass identity.
- Reduce along axis 0, axis 1, and several axes of a 2-D or 3-D tile.
- Build a masked reduction whose denominator is the number of valid lanes, not the padded width.
- Choose tie-breaking and NaN policies explicitly, and pick the neutral element `0`, `1`, `-inf`, or `+inf` for the reduction that follows a masked load.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../../part5_layouts/ch13_memory_optimization/13_complete.ipynb) · [Next →](../ch15_scans/15_complete.ipynb)

**Estimated time:** 65–80 min · **Difficulty:** Intermediate

**Prerequisites:** Chapter 13 (memory optimization); Chapter 7 (elementwise arithmetic); basic statistics (mean and variance).

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

**Thread.** One combine operator, one disappearing axis, one neutral element — shown here as a tree, a matrix, a kept dimension, and a mask.

**Tree.** For $x=(1,2,3,4)$, the sum is a binary tree of depth $\lceil\log_2 N\rceil$: `(1,2,3,4) → (1+2, 3+4) = (3,7) → 3+7 = 10`. The same tree with $\max$ gives $4$; only the combine operator changes. Triton chooses the tree; you choose the operator, axis, and neutral element.

```text
 1   2   3   4
  \ /     \ /
   3       7
    \     /
      10
```

**Axis.** Axis $a$ is the axis that **disappears** from the shape. Reducing axis $1$ walks across each row; reducing axis $0$ walks down each column.

```text
X shape (2,3)                 reduce axis=1 -> shape (2,)
+-----------+                 +-----------+
| 1   2   3 |  row 0  ---->   | 6         |
| 4   5   6 |  row 1  ---->   | 15        |
+-----------+                 +-----------+
      |  |  |
      v  v  v
   reduce axis=0 -> shape (3,)
   [5, 7, 9]
```

**Keepdim.** `(2,)` and `(2,1)` broadcast differently. A row mean used to normalize a matrix needs `(M,1)` so it broadcasts down columns, not `(M,)`.

```text
x                          (2,3)
sum(axis=1)                (2,)      reduced axis removed
sum(axis=1, keepdim=True)  (2,1)     reduced axis kept as 1
sum(axis=0)                (3,)
sum(axis=0, keepdim=True)  (1,3)
sum(all)                   ()
sum(all, keepdim=True)     (1,1)
```

**Rule.** The shape rule is

$$
\operatorname{shape}(y)_k=
\begin{cases}
1, & k=a \text{ and keepdim is true},\\
\text{removed}, & k=a \text{ and keepdim is false},\\
\operatorname{shape}(x)_k, & k\ne a.
\end{cases}
$$

**Mask.** The masked lanes never contribute, because `other` is the neutral element. For a max the neutral element is $-\infty$, for a min $+\infty$, for a product $1$. A masked mean divides by the count of valid lanes, not by the padded width.

```text
x     = [ 1,  2,  9,  9]
mask  = [ 1,  1,  0,  0]
other = [ 1,  2,  0,  0]     sum neutral: 0
sum   = 1 + 2 + 0 + 0 = 3
count = 1 + 1 + 0 + 0 = 2
mean  = 3 / 2 = 1.5
```

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, next_pow2, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

# Part I — Mathematics and concepts

## 14.1 `tl.sum()` — combine lanes with addition

**Connection.** Builds on Chapter 7's elementwise tiles and the concept diagrams above: this is the first reduction, so it sets the axis, shape, and neutral-element pattern every later section reuses.

**Rule.** For $x$ of shape $(d_0,\dots,d_{n-1})$, reducing axis $a$ removes $d_a$; `keepdim=True` in PyTorch or `keep_dims=True` in Triton turns that axis into $1$. A full reduction returns a scalar, or $(1,\dots,1)$ with keepdim.

**Formula.** $y=\sum_{j=0}^{N-1}x_j$.

**Example.** $x=(1,2,3)\to y=6$. On $X=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$, axis $1$ gives $(6,15)$ and axis $0$ gives $(5,7,9)$.

**Where.** Softmax denominators, loss totals, the mean and variance passes, and matrix-multiplication accumulation.

**Triton.** `tl.sum(x, axis=1)`; add `keep_dims=True` when the result must broadcast against the input. A masked load uses `other=0.0`, the add identity.

**Failure.** Loading padded lanes with `other=1.0` adds one per lane; an `int32` sum over more than $2^{31}-1$ elements overflows, so accumulate in `int64` or `float32` first.

**Bridge.** 14.2 keeps the shape rule and swaps addition for the maximum operator.

## 14.2 `tl.max()` — the largest lane

**Connection.** Builds directly on 14.1: the tile, axis, and output shape are unchanged; only the combine operator changes from $+$ to $\max$.

**Rule.** The reduced axis disappears, or becomes $1$ with keepdim; every other axis keeps its size.

**Formula.** $y=\max_{0\le j\lt N}x_j$.

**Example.** $\max(1,5,2)=5$. On $X$, axis $1$ gives $(3,6)$ and axis $0$ gives $(4,5,6)$.

**Where.** Max stabilizes softmax (subtract the row maximum before `exp`), implements max-pooling, and detects the largest score in a classifier.

**Triton.** `tl.max(x, axis=1)`. After a masked load use `tl.where(mask, x, -inf)`, or load with `other=-float("inf")`, so padded lanes cannot win.

**Failure.** `other=0.0` is wrong when every real value is negative: the padded zero becomes the maximum. Triton's reduction default ignores NaN in current releases; filter NaNs explicitly if the policy is "NaN wins".

**Bridge.** 14.3 flips the comparison and asks for the smallest lane instead.

## 14.3 `tl.min()` — the smallest lane

**Connection.** Builds on 14.2 as its opposite: the same tile and shape rule, with the comparison reversed.

**Rule.** The reduced axis disappears, or becomes $1$ with keepdim; the remaining axes are unchanged.

**Formula.** $y=\min_{0\le j\lt N}x_j$.

**Example.** $\min(1,5,2)=1$. On $X$, axis $1$ gives $(1,4)$ and axis $0$ gives $(1,2,3)$.

**Where.** Lower bounds, min-pooling, safe denominators, clipping ranges, and the minimum score in a retrieval set.

**Triton.** `tl.min(x, axis=1)` with `other=float("inf")` on the masked load. The min identity is $+\infty$, not $-\infty$.

**Failure.** Reusing the max neutral element $-\infty$ for a min makes the padded lane win every comparison. Also, `tl.minimum(x, y)` is elementwise; `tl.min(x, axis=...)` is the reduction.

**Bridge.** 14.4 keeps the winning value but returns its index instead.

## 14.4 `tl.argmax()` — where the maximum lives

**Connection.** Builds on 14.2 (the maximum value) and 14.3 (the opposite comparison): the shape is the same as max, but the output is an index.

**Rule.** Argmax has the same output shape as max; the reduced axis disappears (or becomes $1$ with keepdim). The dtype is `int32` in Triton and `int64` in PyTorch.

**Formula.** $y=\arg\max_{0\le j\lt N}x_j$, with $y\in\{0,\dots,N-1\}$.

**Example.** For $(1,5,2)$, $y=1$. On $X$, axis $1$ gives $(2,2)$ and axis $0$ gives $(1,1,1)$.

**Where.** Classification (top-1), retrieval, beam search, and tie detection.

**Triton.** `tl.argmax(x, axis=1)`. The default breaks ties to the left (smallest index); recent releases expose a tie-break flag. Mask padded lanes with $-\infty$.

**Failure.** Treating the returned index as a value; using the sum neutral `0` instead of $-\infty$ for a max; assuming a tie rule that differs from PyTorch's first-index rule.

**Bridge.** 14.5 reverses the direction and returns the index of the minimum.

## 14.5 `tl.argmin()` — where the minimum lives

**Connection.** Builds on 14.4 as its opposite: argmin shares the index contract and reverses the comparison.

**Rule.** Same as min: the reduced axis disappears or becomes $1$ with keepdim; the output is an integer index per remaining lane.

**Formula.** $y=\arg\min_{0\le j\lt N}x_j$, with $y\in\{0,\dots,N-1\}$.

**Example.** For $(1,5,2)$, $y=0$. On $X$, axis $1$ gives $(0,0)$ and axis $0$ gives $(0,0,0)$.

**Where.** Nearest-neighbor search, minimum-loss routing, lower-bound detection, and early-exit decisions.

**Triton.** `tl.argmin(x, axis=1)` with $+\infty$ on padded lanes. The neutral element must match the comparison direction.

**Failure.** Using the argmax neutral $-\infty$ for an argmin; confusing the index with the minimum value; ignoring ties.

**Bridge.** 14.6 leaves indices behind and returns to sums to build statistics.

## 14.6 Mean — sum divided by the true length

**Connection.** Builds on 14.1: mean is the sum reduction followed by a division by the true axis length.

**Rule.** Same as sum: the reduced axis disappears or becomes $1$ with keepdim.

**Formula.** $\mu=\frac1N\sum_{j=0}^{N-1}x_j$.

**Example.** $(1,2,3,4)$ has $\mu=10/4=2.5$. On $X$, axis $1$ gives $(2,5)$ and axis $0$ gives $(2.5,3.5,4.5)$.

**Where.** LayerNorm and BatchNorm statistics, pooled features, loss normalization, and the first pass of variance.

**Triton.** `tl.sum(x, axis=1) / N`, where `N` is the runtime row length. For a masked reduction divide by the count, not by `BLOCK_N`.

**Failure.** Dividing by the padded block width instead of the true length; dividing by zero when a row has no valid lanes (guard with `tl.maximum(count, 1)` and a select).

**Bridge.** 14.7 centers the tile around this mean before squaring.

## 14.7 Variance — the two-pass formula

**Connection.** Builds on 14.6: variance reuses the mean, then averages squared deviations from it.

**Rule.** Same as mean: the reduced axis disappears or becomes $1$; the output is one variance per remaining lane.

**Formula.** Two-pass: $\sigma^2=\frac1N\sum_{j=0}^{N-1}(x_j-\mu)^2$. One-pass identity: $\sigma^2=\left(\frac1N\sum_{j=0}^{N-1}x_j^2\right)-\mu^2$.

**Example.** For $(1,2,3,4)$, $\mu=2.5$ and the centered values are $(-1.5,-0.5,0.5,1.5)$; the squared values sum to $5$, so $\sigma^2=5/4=1.25$. In `float32`, $x=(100001,100002,100003,100004)$ gives $1.25$ by two-pass but about $1024$ by one-pass.

**Where.** Normalization layers, uncertainty estimates, signal-to-noise ratios, and the denominator of a $z$-score.

**Triton.** Two-pass: `mu = tl.sum(x, axis=1)/N`, then `tl.sum((x-mu[:,None])**2, axis=1)/N`. One-pass: compute `sum` and `sumsq` in the same load and use `sumsq/N - mu*mu`.

**Failure — cancellation.** When $\mu$ is large compared with $\sigma$, `sumsq/N` and `mu*mu` are nearly equal and their difference loses most of its significant digits. Prefer two-pass (or Welford) whenever the mean is not near zero.

**Bridge.** 14.8 takes the square root of this variance.

## 14.8 Standard deviation — the square root of variance

**Connection.** Builds on 14.7: standard deviation is the variance followed by a square root.

**Rule.** Same as variance: the reduced axis disappears or becomes $1$ with keepdim.

**Formula.** $\sigma=\sqrt{\sigma^2}=\sqrt{\frac1N\sum_{j=0}^{N-1}(x_j-\mu)^2}$.

**Example.** $(1,2,3,4)$ has $\sigma=\sqrt{1.25}\approx1.118034$. On $X$, axis $1$ gives $\sqrt{2/3}\approx0.816497$; axis $0$ gives $(1.5,1.5,1.5)$.

**Where.** RMSNorm and LayerNorm, error bars, feature scaling, and anomaly scores.

**Triton.** `tl.sqrt(var)`. A variance computed by the one-pass identity can be a tiny negative number after cancellation; clamp with `tl.maximum(var, 0.0)` before the square root.

**Failure.** Forgetting the square root; using the sample variance with $N-1$ when the population value is intended; taking `sqrt` of a negative round-off value, which produces `nan`.

**Bridge.** 14.9 moves from whole-vector statistics to per-axis matrix reductions.

## 14.9 Reduction along axis 0

**Connection.** Builds on 14.1 and the axis diagram: the sum now walks down columns instead of across a whole vector.

**Rule.** Axis $0$ (the row axis) disappears: $(M,N)\to(N,)$, or $(1,N)$ with keepdim.

**Formula.** $Y_j=\sum_{i=0}^{M-1}X_{ij}$, with $X\in\mathbb R^{M\times N}$ and $Y\in\mathbb R^{N}$.

**Example.** For $X=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$, axis $0$ gives $(5,7,9)$; with keepdim the shape is $(1,3)$.

**Where.** BatchNorm reduces over the batch axis; column statistics, per-feature sums, and gradient accumulation over examples do the same.

**Triton.** Load a $[B_M,B_N]$ tile and call `tl.sum(x, axis=0)`; each program owns a column tile. The reduction walks down rows, so the access pattern is strided in row-major memory; keep $B_N$ wide enough to coalesce.

**Failure.** Reducing axis $1$ when the contract says "per column"; writing an output of length $M$ but filling it as if it were length $N$; masking only the last row tile and not the last column tile.

**Bridge.** 14.10 transposes the walk and reduces across each row instead.

## 14.10 Reduction along axis 1

**Connection.** Builds on 14.9: axis $1$ is the contiguous transpose of the axis-0 walk.

**Rule.** Axis $1$ (the column axis) disappears: $(M,N)\to(M,)$, or $(M,1)$ with keepdim.

**Formula.** $Y_i=\sum_{j=0}^{N-1}X_{ij}$, with $X\in\mathbb R^{M\times N}$ and $Y\in\mathbb R^{M}$.

**Example.** The same $X$ gives $(6,15)$ along axis $1$; with keepdim the shape is $(2,1)$.

**Where.** Row-wise softmax, LayerNorm/RMSNorm, per-token statistics, and attention normalization all reduce axis $1$.

**Triton.** One program per row (or per row tile) is the canonical pattern: `cols = tl.arange(0, BLOCK_N)`, load a row, `tl.sum(x, axis=1)`. The reduction walks along contiguous memory.

**Failure.** Using `axis=0` for a row-wise softmax; normalizing with an `(M,)` mean that broadcasts across the wrong dimension instead of an `(M,1)` mean.

**Bridge.** 14.11 composes these one-axis walks into multi-axis reductions.

## 14.11 Multi-dimensional reduction

**Connection.** Builds on 14.9 and 14.10: a multi-axis reduction is these one-axis walks composed in order.

**Rule.** Every reduced axis is removed in order; with keepdim each becomes $1$. Reducing axis $1$ of $(2,2,3)$ gives $(2,3)$; reducing axes $(1,2)$ gives $(2,)$.

**Formula.** For one axis, $Y_{i_1,\dots,i_{k-1},i_{k+1},\dots,i_d}=\bigoplus_{i_k=0}^{n_k-1}X_{i_1,\dots,i_d}$. For all axes, $y=\sum_{i_0}\cdots\sum_{i_{d-1}}X_{i_0,\dots,i_{d-1}}$.

**Example.** Let $A_0=\begin{bmatrix}0&1&2\\3&4&5\end{bmatrix}$ and $A_1=\begin{bmatrix}6&7&8\\9&10&11\end{bmatrix}$ form $A\in\mathbb R^{2\times2\times3}$. Then `sum(axis=1)` gives $\begin{bmatrix}3&5&7\\15&17&19\end{bmatrix}\in\mathbb R^{2\times3}$, `max(axis=1)` gives $\begin{bmatrix}3&4&5\\9&10&11\end{bmatrix}$, and reducing the last axis gives $\begin{bmatrix}3&12\\21&30\end{bmatrix}\in\mathbb R^{2\times2}$.

**Where.** Batched matrix products, multi-head attention (reduce the head axis), per-channel normalization over $(N,H,W)$, and losses over a batch and a sequence.

**Triton.** Reduce one axis at a time, usually the innermost first: `tl.sum(tl.sum(x, axis=1), axis=0)`. Or reshape the tile on the host so the axes you want to collapse are adjacent.

**Failure.** Reducing a subset of the intended axes and silently keeping a size-1 axis; using the wrong axis after a transpose; assuming a 3-D tile keeps the same axis numbering after a reshape.

**Bridge.** 14.12 adds a mask to these multi-axis reductions.

## 14.12 Masked reduction

**Connection.** Builds on 14.6 (mean) and 14.10 (row reduction): the same statistics, with a mask deciding which lanes count.

**Rule.** The mask has the same shape as the tile it guards. The reduction output has the usual reduced-axis shape; the count is a reduction of the mask to that same shape.

**Formula.** $y=\sum_{j=0}^{N-1}m_jx_j$ and $\bar x=\frac{\sum_j m_jx_j}{\sum_j m_j}$, with $m_j\in\{0,1\}$.

**Example.** For $x=(1,2,9,9)$ and $m=(1,1,0,0)$, the masked sum is $3$, the count is $2$, and the masked mean is $1.5$. The two masked values never enter the sum.

**Where.** Variable-length sequences, padding tokens, `ignore_index` in a loss, missing features, and attention masks.

**Triton.** `x = tl.load(x_ptr + offs, mask=m, other=0.0)`; `total = tl.sum(tl.where(m, x, 0.0), axis=1)`; `count = tl.sum(m.to(tl.float32), axis=1)`; `mean = tl.where(count > 0, total / tl.maximum(count, 1.0), 0.0)`. The `other` value and the `tl.where` value must both be neutral for the operation that follows.

**Failure.** Loading masked lanes with `other=1.0` and summing; dividing by `BLOCK_N` instead of the count; computing a max with `other=0.0` when the valid values are negative; forgetting that `count=0` makes the mean `0/0`.

**Bridge.** The chapter map and worked example consolidate these twelve rules into one table and one numerical check.

**Where this leads.** Part II implements the same rules as loops, Part III as Triton kernels, and Part IV fuses and masks them. Chapter 15 then keeps one output per prefix instead of one output per slice.

## Chapter map

**Connection.** These twelve rows are the point-wise summary of Part I; the worked example below checks them numerically.

| § | Reduction | Formula | Output shape | Kernel/demo |
|---|---|---|---|---|
| **14.1** | `tl.sum()` | $y=\sum_j x_j$ | reduced axis removed, or $1$ | 1-D sum kernel; row sum |
| **14.2** | `tl.max()` | $y=\max_j x_j$ | same as sum | 1-D max kernel; row max |
| **14.3** | `tl.min()` | $y=\min_j x_j$ | same as sum | 1-D min kernel; row min |
| **14.4** | `tl.argmax()` | $y=\arg\max_j x_j$ | same as max, dtype `int32` | 1-D argmax kernel; row argmax |
| **14.5** | `tl.argmin()` | $y=\arg\min_j x_j$ | same as min, dtype `int32` | 1-D argmin kernel |
| **14.6** | Mean | $\mu=\frac1N\sum_j x_j$ | same as sum | Fused moments |
| **14.7** | Variance | $\sigma^2=\frac1N\sum_j(x_j-\mu)^2$ | same as mean | Fused moments; two-pass check |
| **14.8** | Standard deviation | $\sigma=\sqrt{\sigma^2}$ | same as variance | Fused moments |
| **14.9** | Axis 0 | $Y_j=\sum_i X_{ij}$ | $(M,N)\to(N,)$ or $(1,N)$ | Column-tile moments kernel |
| **14.10** | Axis 1 | $Y_i=\sum_j X_{ij}$ | $(M,N)\to(M,)$ or $(M,1)$ | Row-tile reduction kernel |
| **14.11** | Multi-dimensional | $\bigoplus_{i_k}X_{\dots}$ | remove (or keep as $1$) every reduced axis | Two-axis composition; 3-D checks |
| **14.12** | Masked reduction | $y=\sum_j m_jx_j$ | reduced-axis shape; count likewise | Masked mean kernel |

## Worked numerical example

**Connection.** Every reduction from 14.1–14.12 appears once below; the tables are the hand-checkable version of the chapter map.

**Setup.** $X=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}\in\mathbb R^{2\times3}$.

### Axis 0 — collapse the rows

**Rule.** Each column is reduced over the two rows, so the output shape is $(3,)$; keepdim gives $(1,3)$.

| Reduction | Result | Shape | Keepdim shape |
|---|---|---|---|
| sum | $(5,7,9)$ | $(3,)$ | $(1,3)$ |
| max | $(4,5,6)$ | $(3,)$ | $(1,3)$ |
| min | $(1,2,3)$ | $(3,)$ | $(1,3)$ |
| argmax | $(1,1,1)$ | $(3,)$ | $(1,3)$ |
| argmin | $(0,0,0)$ | $(3,)$ | $(1,3)$ |
| mean | $\left(\frac52,\frac72,\frac92\right)$ | $(3,)$ | $(1,3)$ |
| variance | $\left(\frac94,\frac94,\frac94\right)$ | $(3,)$ | $(1,3)$ |
| std | $\left(\frac32,\frac32,\frac32\right)$ | $(3,)$ | $(1,3)$ |

**Hand check, column 1.** $2+5=7$; mean $=7/2=3.5$; variance $=\frac{(2-3.5)^2+(5-3.5)^2}{2}=2.25$; std $=1.5$.

### Axis 1 — collapse the columns

**Rule.** Each row is reduced over the three columns, so the output shape is $(2,)$; keepdim gives $(2,1)$.

| Reduction | Result | Shape | Keepdim shape |
|---|---|---|---|
| sum | $(6,15)$ | $(2,)$ | $(2,1)$ |
| max | $(3,6)$ | $(2,)$ | $(2,1)$ |
| min | $(1,4)$ | $(2,)$ | $(2,1)$ |
| argmax | $(2,2)$ | $(2,)$ | $(2,1)$ |
| argmin | $(0,0)$ | $(2,)$ | $(2,1)$ |
| mean | $(2,5)$ | $(2,)$ | $(2,1)$ |
| variance | $\left(\frac23,\frac23\right)$ | $(2,)$ | $(2,1)$ |
| std | $\left(\sqrt{\frac23},\sqrt{\frac23}\right)$ | $(2,)$ | $(2,1)$ |

**Hand check, row 0.** $1+2+3=6$; mean $=2$; the centered values are $(-1,0,1)$; variance $=\frac{1+0+1}{3}=\frac23$; std $=\sqrt{2/3}\approx0.816497$.

### A $2\times2\times3$ reduction to $2\times3$

**Setup.** $A_0=\begin{bmatrix}0&1&2\\3&4&5\end{bmatrix}$ and $A_1=\begin{bmatrix}6&7&8\\9&10&11\end{bmatrix}$ form $A\in\mathbb R^{2\times2\times3}$.

**Rule.** Reducing axis $1$ (the middle axis) leaves shape $(2,3)$:

$$
\sum_{i_1=0}^{1}A_{i_0,i_1,i_2}
=
\begin{bmatrix}3&5&7\\15&17&19\end{bmatrix},
\qquad
\max_{i_1}A_{i_0,i_1,i_2}
=
\begin{bmatrix}3&4&5\\9&10&11\end{bmatrix}.
$$

**Rule.** Reducing the last axis instead gives shape $(2,2)$: $\sum_{i_2=0}^{2}A_{i_0,i_1,i_2}=\begin{bmatrix}3&12\\21&30\end{bmatrix}$.

The next cell verifies every number and every shape above with assertions only.

In [ ]:
X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
assert X.shape == (2, 3)

torch.testing.assert_close(X.sum(0), torch.tensor([5.0, 7.0, 9.0]))
torch.testing.assert_close(X.max(0).values, torch.tensor([4.0, 5.0, 6.0]))
torch.testing.assert_close(X.min(0).values, torch.tensor([1.0, 2.0, 3.0]))
assert X.argmax(0).tolist() == [1, 1, 1]
assert X.argmin(0).tolist() == [0, 0, 0]
torch.testing.assert_close(X.mean(0), torch.tensor([2.5, 3.5, 4.5]))
torch.testing.assert_close(X.var(0, unbiased=False), torch.full((3,), 2.25))
torch.testing.assert_close(X.std(0, unbiased=False), torch.full((3,), 1.5))

torch.testing.assert_close(X.sum(1), torch.tensor([6.0, 15.0]))
torch.testing.assert_close(X.max(1).values, torch.tensor([3.0, 6.0]))
torch.testing.assert_close(X.min(1).values, torch.tensor([1.0, 4.0]))
assert X.argmax(1).tolist() == [2, 2]
assert X.argmin(1).tolist() == [0, 0]
torch.testing.assert_close(X.mean(1), torch.tensor([2.0, 5.0]))
torch.testing.assert_close(X.var(1, unbiased=False), torch.full((2,), 2.0 / 3.0))
torch.testing.assert_close(X.std(1, unbiased=False), torch.full((2,), (2.0 / 3.0) ** 0.5))

assert X.sum(1).shape == (2,) and X.sum(1, keepdim=True).shape == (2, 1)
assert X.sum(0).shape == (3,) and X.sum(0, keepdim=True).shape == (1, 3)
assert X.sum().shape == () and X.sum(dim=(0, 1), keepdim=True).shape == (1, 1)

v = torch.tensor([1.0, 2.0, 3.0, 4.0])
assert sum(range(1, 5)) == 10 and max(1, 2, 3, 4) == 4
assert torch.tensor([1.0, 5.0, 2.0]).max().item() == 5.0
assert torch.tensor([1.0, 5.0, 2.0]).min().item() == 1.0
assert torch.tensor([1.0, 5.0, 2.0]).argmax().item() == 1
assert torch.tensor([1.0, 5.0, 2.0]).argmin().item() == 0
torch.testing.assert_close(v.mean(), torch.tensor(2.5))
torch.testing.assert_close(v.var(unbiased=False), torch.tensor(1.25))
torch.testing.assert_close(v.std(unbiased=False), torch.tensor(1.25 ** 0.5))

big = torch.tensor([100001.0, 100002.0, 100003.0, 100004.0])
one_pass = (big * big).mean() - big.mean() ** 2
two_pass = ((big - big.mean()) ** 2).mean()
assert abs(two_pass.item() - 1.25) < 1e-6
assert one_pass.item() > 1000.0

A = torch.arange(12.0, dtype=torch.float32).reshape(2, 2, 3)
assert A.shape == (2, 2, 3)
torch.testing.assert_close(A.sum(1), torch.tensor([[3.0, 5.0, 7.0], [15.0, 17.0, 19.0]]))
torch.testing.assert_close(A.max(1).values, torch.tensor([[3.0, 4.0, 5.0], [9.0, 10.0, 11.0]]))
torch.testing.assert_close(A.sum(-1), torch.tensor([[3.0, 12.0], [21.0, 30.0]]))
assert A.sum(1).shape == (2, 3) and A.sum(-1).shape == (2, 2)

# Part II — Method 01: naive Python/PyTorch reference

**Connection.** Builds on Part I's definitions: the loops below implement exactly the formula and shape rules just stated, with no vectorization.

**Rule.** One Python iteration per element; the output drops the reduced axis.

**Reference functions.**

1. `naive_sum` — add every lane along the chosen axis.
2. `naive_extreme` — scan for max, min, argmax, or argmin.
3. `naive_mean` — divide the naive sum by the axis length.
4. `naive_var` — center by the naive mean, square, and average the squares.

**Example.** The tests use the $2\times3$ worked example and a random $7\times5$ matrix so that a wrong axis cannot pass by symmetry.

**Bridge.** Part III keeps the same contract but moves the loop into a Triton program.

In [ ]:
def naive_sum(x: torch.Tensor, axis: int) -> torch.Tensor:
    '''Method 01: two nested loops, one addition per input element.'''
    m, n = x.shape
    axis = axis % 2
    if axis == 0:
        out = torch.zeros(n, dtype=x.dtype)
        for j in range(n):
            for i in range(m):
                out[j] += x[i, j]
    else:
        out = torch.zeros(m, dtype=x.dtype)
        for i in range(m):
            for j in range(n):
                out[i] += x[i, j]
    return out


def naive_extreme(x: torch.Tensor, axis: int, op: str) -> torch.Tensor:
    '''Method 01: max, min, argmax, or argmin with an explicit scan.'''
    m, n = x.shape
    axis = axis % 2
    size = n if axis == 0 else m
    arg = op.startswith("arg")
    out = torch.empty(size, dtype=torch.int64 if arg else x.dtype)
    for k in range(size):
        lane = x[:, k] if axis == 0 else x[k, :]
        best = int(torch.argmax(lane) if op in ("argmax", "max") else torch.argmin(lane))
        out[k] = best if arg else lane[best]
    return out


def naive_mean(x: torch.Tensor, axis: int) -> torch.Tensor:
    '''Method 01: mean is the naive sum divided by the true axis length.'''
    return naive_sum(x, axis) / x.shape[axis % 2]


def naive_var(x: torch.Tensor, axis: int) -> torch.Tensor:
    '''Method 01: population variance by the two-pass formula.'''
    axis = axis % 2
    mu = naive_mean(x, axis)
    centered = x - mu.unsqueeze(axis)
    return naive_sum(centered * centered, axis) / x.shape[axis]

In [ ]:
def test_method01() -> None:
    torch.manual_seed(0)
    for x in (torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]), torch.randn(7, 5)):
        for axis in (0, 1):
            torch.testing.assert_close(naive_sum(x, axis), x.sum(axis))
            torch.testing.assert_close(naive_extreme(x, axis, "max"), x.max(axis).values)
            torch.testing.assert_close(naive_extreme(x, axis, "min"), x.min(axis).values)
            assert naive_extreme(x, axis, "argmax").tolist() == x.argmax(axis).tolist()
            assert naive_extreme(x, axis, "argmin").tolist() == x.argmin(axis).tolist()
            torch.testing.assert_close(naive_mean(x, axis), x.mean(axis))
            torch.testing.assert_close(naive_var(x, axis), x.var(axis, unbiased=False))
    print("method 01: loops match PyTorch on the worked example and a random 7x5 matrix")


test_method01()

# Part III — Method 02: Triton reduction kernels

## 1-D kernels: one program, one vector

**Connection.** Builds on Part II's reference and Part I's neutral-element table: the kernel computes the same reduction, but lanes are a tile.

**Rule.** One program owns the whole vector; the block size is the next power of two above $N$, so a tail mask is still required.

**Formula.** With `offs = {0,...,B-1}` and `mask = offs < N`, $y=\bigoplus_{j=0}^{N-1}x_j$.

**Triton.** The operator is a `tl.constexpr` selector (`SUM`, `MAX`, `MIN`); argmax and argmin need an integer output, so they use a second kernel. The load uses `other=0.0` for sum, `-inf` for max/argmax, and `+inf` for min/argmin.

**Failure.** This one-block kernel assumes $N\le B$; a production kernel loops over blocks and combines partial results. The test cell also runs `keepdim_sum_kernel` to confirm `keep_dims=True` returns shape `(1,)`, not a scalar.

**Bridge.** The next subsection keeps the same contract but makes the reduction axis a row tile.

In [ ]:
REDUCE_SUM, REDUCE_MAX, REDUCE_MIN = 0, 1, 2
ARGMAX, ARGMIN = 0, 1


@triton.jit
def reduce_1d_kernel(x_ptr, out_ptr, n, OP: tl.constexpr, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n
    if OP == REDUCE_MAX:
        x = tl.load(x_ptr + offs, mask=mask, other=-float("inf"))
        result = tl.max(x, axis=0)
    elif OP == REDUCE_MIN:
        x = tl.load(x_ptr + offs, mask=mask, other=float("inf"))
        result = tl.min(x, axis=0)
    else:
        x = tl.load(x_ptr + offs, mask=mask, other=0.0)
        result = tl.sum(x, axis=0)
    tl.store(out_ptr, result)


@triton.jit
def arg_reduce_1d_kernel(x_ptr, out_ptr, n, OP: tl.constexpr, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n
    if OP == ARGMAX:
        x = tl.load(x_ptr + offs, mask=mask, other=-float("inf"))
        result = tl.argmax(x, axis=0)
    else:
        x = tl.load(x_ptr + offs, mask=mask, other=float("inf"))
        result = tl.argmin(x, axis=0)
    tl.store(out_ptr, result)


def reduce_1d(x: torch.Tensor, op: int, block: int | None = None) -> torch.Tensor:
    x = x.contiguous()
    block = block or next_pow2(x.numel())
    out = torch.empty(1, device=x.device, dtype=x.dtype)
    reduce_1d_kernel[(1,)](x, out, x.numel(), OP=op, BLOCK=block)
    return out[0]


def arg_reduce_1d(x: torch.Tensor, op: int, block: int | None = None) -> torch.Tensor:
    x = x.contiguous()
    block = block or next_pow2(x.numel())
    out = torch.empty(1, device=x.device, dtype=torch.int32)
    arg_reduce_1d_kernel[(1,)](x, out, x.numel(), OP=op, BLOCK=block)
    return out[0]


@triton.jit
def keepdim_sum_kernel(x_ptr, out_ptr, n, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    x = tl.load(x_ptr + offs, mask=offs < n, other=0.0)
    # keep_dims=True keeps the reduced axis as a size-1 axis.
    tl.store(out_ptr + tl.arange(0, 1), tl.sum(x, axis=0, keep_dims=True))

In [ ]:
def test_reduce_1d() -> None:
    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE)
    torch.testing.assert_close(reduce_1d(x, REDUCE_SUM), x.sum())
    torch.testing.assert_close(reduce_1d(x, REDUCE_MAX), x.max())
    torch.testing.assert_close(reduce_1d(x, REDUCE_MIN), x.min())
    assert arg_reduce_1d(x, ARGMAX).item() == x.argmax().item()
    assert arg_reduce_1d(x, ARGMIN).item() == x.argmin().item()

    # A tile whose real values are all negative: other=0 would be a wrong max.
    neg = -x.abs() - 1.0
    torch.testing.assert_close(reduce_1d(neg, REDUCE_MAX), neg.max())
    torch.testing.assert_close(reduce_1d(neg, REDUCE_MIN), neg.min())

    ties = torch.tensor([1.0, 3.0, 3.0, 2.0], device=DEVICE)
    assert arg_reduce_1d(ties, ARGMAX).item() == 1
    assert arg_reduce_1d(ties, ARGMIN).item() == 0

    kept = torch.empty(1, device=DEVICE, dtype=x.dtype)
    keepdim_sum_kernel[(1,)](x, kept, x.numel(), BLOCK=next_pow2(x.numel()))
    assert kept.shape == (1,)
    torch.testing.assert_close(kept[0], x.sum())
    print("method 02: 1-D reductions match PyTorch; keep_dims keeps the size-1 axis")


run_if_gpu(test_reduce_1d)

## 2-D row reduction: one program per row tile

**Connection.** Builds on the 1-D kernel and 14.10: the row is now a tile, and axis $1$ is the combine direction.

**Rule.** A program owns `rows = pid * B_M + {0,...,B_M-1}` and `cols = {0,...,B_N-1}`, with `mask = rows < M and cols < N` and `grid = ceil(M / B_M)`.

**Example.** Tile map for $M=5$, $N=3$, $B_M=2$, $B_N=4$:

$$
\begin{array}{c|c|c|l}
\texttt{pid} & \text{rows} & \text{live cols} & \text{padded lanes}\\
\hline
0 & 0,1 & 0,1,2 & \text{col }3\\
1 & 2,3 & 0,1,2 & \text{col }3\\
2 & 4,5 & 0,1,2 & \text{row }5,\ \text{col }3
\end{array}
$$

**Triton.** Program 2 owns one real row and one padded row; the row mask suppresses the stores for row 5. The column mask makes `cols=3` neutral: `0` for the sum, $-\infty$ for the max and argmax, $+\infty$ for the min. The kernel loads with `other=0.0` once, then builds three masked views with `tl.where`; one neutral would be wrong for at least one of the four reductions.

**Failure.** A single `other` for all reductions; forgetting the row or column mask.

**Bridge.** Part IV fuses these reductions into per-row statistics.

In [ ]:
@triton.jit
def row_reduce_kernel(x_ptr, sum_ptr, max_ptr, min_ptr, argmax_ptr, M, N,
                      BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    rows = tl.program_id(0) * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)
    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)

    zero = tl.where(mask, x, 0.0)
    big = tl.where(mask, x, -float("inf"))
    small = tl.where(mask, x, float("inf"))
    row_mask = rows < M

    tl.store(sum_ptr + rows, tl.sum(zero, axis=1), mask=row_mask)
    tl.store(max_ptr + rows, tl.max(big, axis=1), mask=row_mask)
    tl.store(min_ptr + rows, tl.min(small, axis=1), mask=row_mask)
    tl.store(argmax_ptr + rows, tl.argmax(big, axis=1), mask=row_mask)


def row_reduce(x: torch.Tensor, BLOCK_M: int = 8, BLOCK_N: int | None = None):
    M, N = x.shape
    BLOCK_N = BLOCK_N or next_pow2(N)
    x = x.contiguous()
    s = torch.empty(M, device=x.device, dtype=x.dtype)
    mx = torch.empty(M, device=x.device, dtype=x.dtype)
    mn = torch.empty(M, device=x.device, dtype=x.dtype)
    am = torch.empty(M, device=x.device, dtype=torch.int32)
    row_reduce_kernel[(triton.cdiv(M, BLOCK_M),)](
        x, s, mx, mn, am, M, N, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N
    )
    return s, mx, mn, am

In [ ]:
def test_row_reduce() -> None:
    torch.manual_seed(0)
    x = torch.randn(29, 37, device=DEVICE)
    s, mx, mn, am = row_reduce(x)
    torch.testing.assert_close(s, x.sum(dim=1))
    torch.testing.assert_close(mx, x.max(dim=1).values)
    torch.testing.assert_close(mn, x.min(dim=1).values)
    assert am.tolist() == x.argmax(dim=1).tolist()

    # The exact 2x3 worked example, with a padded block of width 4.
    tiny = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], device=DEVICE)
    s, mx, mn, am = row_reduce(tiny, BLOCK_M=2, BLOCK_N=4)
    torch.testing.assert_close(s, torch.tensor([6.0, 15.0], device=DEVICE))
    torch.testing.assert_close(mx, torch.tensor([3.0, 6.0], device=DEVICE))
    torch.testing.assert_close(mn, torch.tensor([1.0, 4.0], device=DEVICE))
    assert am.tolist() == [2, 2]
    print("method 02: 2-D row kernel matches the 2x3 worked example and a 29x37 tail")


run_if_gpu(test_row_reduce)

# Part IV — Method 03: fused moments and masked statistics

## Fused sum-of-squares, mean, and variance along axis 1

**Connection.** Builds on 14.6–14.8 and the row kernel above: the same row statistics, computed from one load.

**Rule.** A normalization layer needs $S_i=\sum_j x_{ij}$, $Q_i=\sum_j x_{ij}^2$, $\mu_i=S_i/N$, and $\sigma_i^2=\max(Q_i/N-\mu_i^2,0)$. Each output is `(M,)`; keepdim gives `(M,1)`.

**Formula.** Two-pass alternative: $\sigma_i^2=\frac1N\sum_j(x_{ij}-\mu_i)^2$.

**Where.** LayerNorm/RMSNorm statistics; the clamp prevents a tiny negative variance from becoming `nan` after `sqrt`.

**Triton.** One load, two sums in registers, three stores; no centered copy of the matrix.

**Failure.** One-pass cancellation when the mean is large; forgetting the clamp.

**Bridge.** The next subsection transposes the walk to axis $0$.

In [ ]:
@triton.jit
def moments_axis1_kernel(x_ptr, mean_ptr, var_ptr, sumsq_ptr, M, N,
                         BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    rows = tl.program_id(0) * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)
    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    valid = tl.where(mask, x, 0.0)

    total = tl.sum(valid, axis=1)
    sumsq = tl.sum(valid * valid, axis=1)
    mean = total / N
    var = tl.maximum(sumsq / N - mean * mean, 0.0)

    row_mask = rows < M
    tl.store(mean_ptr + rows, mean, mask=row_mask)
    tl.store(var_ptr + rows, var, mask=row_mask)
    tl.store(sumsq_ptr + rows, sumsq, mask=row_mask)


def moments_axis1(x: torch.Tensor, BLOCK_M: int = 8, BLOCK_N: int | None = None):
    M, N = x.shape
    BLOCK_N = BLOCK_N or next_pow2(N)
    x = x.contiguous()
    mean = torch.empty(M, device=x.device, dtype=x.dtype)
    var = torch.empty(M, device=x.device, dtype=x.dtype)
    sumsq = torch.empty(M, device=x.device, dtype=x.dtype)
    moments_axis1_kernel[(triton.cdiv(M, BLOCK_M),)](
        x, mean, var, sumsq, M, N, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N
    )
    return mean, var, sumsq

In [ ]:
def test_moments_axis1() -> None:
    torch.manual_seed(0)
    x = torch.randn(29, 37, device=DEVICE)
    mean, var, sumsq = moments_axis1(x)
    torch.testing.assert_close(mean, x.mean(dim=1))
    torch.testing.assert_close(var, x.var(dim=1, unbiased=False), atol=1e-4, rtol=1e-4)
    torch.testing.assert_close(sumsq, (x * x).sum(dim=1), atol=1e-4, rtol=1e-4)

    tiny = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], device=DEVICE)
    mean, var, sumsq = moments_axis1(tiny, BLOCK_M=2, BLOCK_N=4)
    torch.testing.assert_close(mean, torch.tensor([2.0, 5.0], device=DEVICE))
    torch.testing.assert_close(var, torch.full((2,), 2.0 / 3.0, device=DEVICE))
    torch.testing.assert_close(sumsq, torch.tensor([14.0, 77.0], device=DEVICE))
    print("method 03: fused mean/variance/sum-of-squares match axis=1 references")


run_if_gpu(test_moments_axis1)

## Axis-0 variant

**Connection.** Builds on the axis-1 moments kernel; only the tile orientation changes.

**Rule.** A program owns a column tile of width $B_N$; the full row axis is $B_M=\text{next\_pow2}(M)$. Output shape is `(N,)`; keepdim gives `(1,N)`.

**Formula.** $S_j=\sum_i x_{ij}$, $\mu_j=S_j/M$, and $\sigma_j^2=\max(Q_j/M-\mu_j^2,0)$.

**Where.** BatchNorm: one mean and one variance per feature over the batch. The combine walks down rows, so a wider $B_N$ tile is usually better.

**Failure.** The illustrative kernel loads the whole row axis at once; for a batch larger than the register budget, loop over row tiles and accumulate partial sums and sums of squares before dividing.

**Bridge.** 14.12 adds a mask to these statistics.

In [ ]:
@triton.jit
def moments_axis0_kernel(x_ptr, mean_ptr, var_ptr, M, N,
                         BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    rows = tl.arange(0, BLOCK_M)
    cols = tl.program_id(0) * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)
    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    valid = tl.where(mask, x, 0.0)

    total = tl.sum(valid, axis=0)
    sumsq = tl.sum(valid * valid, axis=0)
    mean = total / M
    var = tl.maximum(sumsq / M - mean * mean, 0.0)

    col_mask = cols < N
    tl.store(mean_ptr + cols, mean, mask=col_mask)
    tl.store(var_ptr + cols, var, mask=col_mask)


def moments_axis0(x: torch.Tensor, BLOCK_N: int = 16):
    M, N = x.shape
    x = x.contiguous()
    mean = torch.empty(N, device=x.device, dtype=x.dtype)
    var = torch.empty(N, device=x.device, dtype=x.dtype)
    moments_axis0_kernel[(triton.cdiv(N, BLOCK_N),)](
        x, mean, var, M, N, BLOCK_M=next_pow2(M), BLOCK_N=BLOCK_N
    )
    return mean, var

In [ ]:
def test_moments_axis0() -> None:
    torch.manual_seed(0)
    x = torch.randn(17, 37, device=DEVICE)
    mean, var = moments_axis0(x)
    torch.testing.assert_close(mean, x.mean(dim=0))
    torch.testing.assert_close(var, x.var(dim=0, unbiased=False), atol=1e-4, rtol=1e-4)

    tiny = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], device=DEVICE)
    mean, var = moments_axis0(tiny, BLOCK_N=4)
    torch.testing.assert_close(mean, torch.tensor([2.5, 3.5, 4.5], device=DEVICE))
    torch.testing.assert_close(var, torch.full((3,), 2.25, device=DEVICE))
    print("method 03: axis=0 moments match the column references")


run_if_gpu(test_moments_axis0)

## Masked mean: divide by the count, not by the width

**Connection.** Builds on 14.6 mean and 14.12: the denominator is now a mask count.

**Rule.** For a padded row with true length $\ell$, $\bar x_i=\frac{\sum_{j\lt \ell}x_{ij}}{\sum_{j\lt \ell}1}=\frac{\sum_j m_{ij}x_{ij}}{\sum_j m_{ij}}$, with $m_{ij}=[j\lt \ell]$. Output shape is `(M,)`, or `(M,1)` with keepdim.

**Triton.** The sum uses `other=0.0`; the count is the sum of the mask itself. A row with no valid lane returns `0.0`, not `0/0`.

**Failure.** Dividing by `BLOCK_N` shrinks every mean by $\ell/B_N$; dividing by padded `N` is also wrong. Only the mask count is correct for every row.

**Bridge.** The policy section fixes the remaining choices: ties, NaN, and neutral elements.

In [ ]:
@triton.jit
def masked_mean_kernel(x_ptr, out_ptr, M, N, valid_to,
                       BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    rows = tl.program_id(0) * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = tl.arange(0, BLOCK_N)
    in_range = (rows[:, None] < M) & (cols[None, :] < N)
    keep = in_range & (cols[None, :] < valid_to)
    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :], mask=in_range, other=0.0)

    total = tl.sum(tl.where(keep, x, 0.0), axis=1)
    count = tl.sum(keep.to(tl.float32), axis=1)
    mean = tl.where(count > 0.0, total / tl.maximum(count, 1.0), 0.0)
    tl.store(out_ptr + rows, mean, mask=rows < M)


def masked_mean(x: torch.Tensor, valid_to: int | None = None,
                BLOCK_M: int = 8, BLOCK_N: int | None = None) -> torch.Tensor:
    M, N = x.shape
    valid_to = N if valid_to is None else valid_to
    BLOCK_N = BLOCK_N or next_pow2(N)
    x = x.contiguous()
    out = torch.empty(M, device=x.device, dtype=x.dtype)
    masked_mean_kernel[(triton.cdiv(M, BLOCK_M),)](
        x, out, M, N, valid_to, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N
    )
    return out

In [ ]:
def test_masked_mean() -> None:
    torch.manual_seed(0)
    x = torch.randn(19, 37, device=DEVICE)
    torch.testing.assert_close(masked_mean(x), x.mean(dim=1))
    torch.testing.assert_close(masked_mean(x, 5), x[:, :5].mean(dim=1))

    # The text example: only the first two columns are valid.
    tiny = torch.tensor([[1.0, 2.0, 9.0, 9.0], [3.0, 4.0, 8.0, 8.0]], device=DEVICE)
    got = masked_mean(tiny, 2, BLOCK_M=2, BLOCK_N=4)
    torch.testing.assert_close(got, torch.tensor([1.5, 3.5], device=DEVICE))

    # An all-masked row returns 0, not nan.
    none_valid = masked_mean(tiny, 0, BLOCK_M=2, BLOCK_N=4)
    torch.testing.assert_close(none_valid, torch.zeros(2, device=DEVICE))
    print("method 03: masked mean divides by the valid count, not by the padded width")


run_if_gpu(test_masked_mean)

# Ties, NaN policy, and neutral elements

**Connection.** Builds on 14.4–14.5 and 14.12: once indices and masks exist, tie and NaN rules become part of the contract.

## Argmax/argmin ties

**Rule.** Both PyTorch and Triton break ties at the **first** (leftmost) occurrence by default: $\arg\max_j x_j=\min\{j:x_j=\max_k x_k\}$.

**Example.** For $x=(1,2,2,1)$, `argmax` returns $1$ and `argmin` returns $0$.

**Triton.** `tl.argmax` exposes a tie-break flag in recent releases; do not rely on the rightmost index unless you pass it.

## NaN policy

**Rule.** PyTorch's `max`/`min` **propagate** NaN; Triton's reduction default **ignores** NaN in current releases (the `propagate_nan` policy can be set explicitly). The libraries disagree on the same input, so pick one policy.

- **NaN wins:** check `torch.isnan(result)` and handle it at the host.
- **NaN is missing:** filter before the reduction with `clean = tl.where(x == x, x, -inf)` for a max, or `+inf` for a min.
- **NaN is an error:** assert `torch.isfinite(x).all()` before the kernel.

**Triton.** `x == x` is false exactly for NaN, so it is a portable filter; `tl.isnan` is also available in recent releases.

## Neutral elements

**Rule.** The neutral element belongs to the **combine operator**, not to the dtype.

| Reduction | Neutral `other` | Identity |
|---|---|---|
| sum | `0.0` | $x+0=x$ |
| product | `1.0` | $x\cdot1=x$ |
| max | `-inf` | $\max(x,-\infty)=x$ |
| min | `+inf` | $\min(x,+\infty)=x$ |
| count | `0` | adds nothing |
| argmax | `-inf` | padded lane cannot win |
| argmin | `+inf` | padded lane cannot win |

**Failure.** A masked max loaded with `other=0.0` is wrong whenever all valid values are negative.

**Bridge.** The pitfalls table and self-check turn these rules into failure modes.

In [ ]:
def test_policy() -> None:
    tied = torch.tensor([[1.0, 2.0, 2.0, 1.0], [3.0, 3.0, 1.0, 1.0]])
    assert tied.argmax(dim=1).tolist() == [1, 0]
    assert tied.argmin(dim=1).tolist() == [0, 2]

    nan_x = torch.tensor([1.0, float("nan"), 3.0])
    assert torch.isnan(nan_x.max())
    assert torch.isnan(nan_x.min())
    clean = torch.where(nan_x == nan_x, nan_x, torch.tensor(float("-inf")))
    assert clean.max().item() == 3.0

    s = torch.tensor([1.0, -2.0, 3.0])
    assert torch.equal(s + 0.0, s)
    assert torch.equal(s * 1.0, s)
    assert torch.full_like(s, float("-inf")).maximum(s).equal(s)
    assert torch.full_like(s, float("inf")).minimum(s).equal(s)
    print("policy: first-index ties, NaN filtering, and neutral elements match the text")


test_policy()

# Pitfalls, self-check, and summary

**Connection.** Builds on every section: the table names the failure mode, the self-check asks you to reproduce the rule, and the summary restates the chain.

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| Axis confusion | `axis=0` and `axis=1` swap row and column statistics | Write the output shape first; axis $a$ is the one that disappears |
| Forgetting keepdim | `(M,)` broadcasts across the wrong dimension | Use `(M,1)` (`keepdim=True` / `keep_dims=True`) for row statistics |
| Wrong masked `other` | Padded lanes win a max or inflate a sum | `0` for sum, `-inf` for max, `+inf` for min, `1` for product |
| NaN policy | PyTorch propagates, Triton may ignore | Filter with `x == x` or assert finiteness |
| Variance cancellation | `E[x^2]-mu^2` loses digits when `mu` is large | Two-pass or Welford; clamp before `sqrt` |
| Integer overflow | `int32` sum wraps past $2^{31}-1$ | Accumulate in `int64` or `float32` |
| Wrong reduction axis | Shape mismatch or a plausible but wrong number | Derive the shape from the contract, then test an asymmetric matrix |
| `argmax` as a value | An index is used where the maximum was intended | Use `tl.max` for the value, `tl.argmax` for the index |
| Dividing by `BLOCK_N` | A masked mean is scaled by `valid/BLOCK_N` | Divide by the mask count |
| Empty mask | `0/0 = nan` | `tl.where(count > 0, total / max(count,1), 0)` |

## Self-check

1. Write the reduction tree for $(1,2,3,4)$ with sum and with max.
2. For $X=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$, list the eight axis-0 results and the eight axis-1 results, including output shapes.
3. Why is `X.sum(1)` of shape `(2,)` while `X.sum(1, keepdim=True)` is `(2,1)`? Which one broadcasts correctly over a `(2,3)` matrix?
4. Reduce a $2\times2\times3$ tensor along axis $1$. What is the output shape, and what does axis $2$ give?
5. Why does a masked max need `other=-inf` rather than `other=0`?
6. Derive the two-pass variance and explain the cancellation in the one-pass form.
7. What neutral element does each of sum, product, max, min, and count use?
8. What tie rule do PyTorch and Triton use for `argmax`, and what happens to NaN?
9. Why is the best row-wise kernel one program per row rather than one program per element?
10. Given a padded row of length $\ell$ stored in a block of width $B_N$, write the masked mean.

## Summary

- A reduction collapses one axis (or several); the reduced axis disappears, or becomes $1$ with keepdim.
- `tl.sum`, `tl.max`, `tl.min`, `tl.argmax`, and `tl.argmin` differ only in the combine operator and output dtype.
- Mean is sum divided by the true length; variance has a stable two-pass form and a cancellation-prone one-pass form; standard deviation is its square root.
- Axis $0$ walks down columns; axis $1$ walks across rows; multi-dimensional reductions compose one axis at a time.
- A masked reduction needs the neutral element of the combine operator and a denominator equal to the count of valid lanes.
- Ties and NaN are policy decisions, not accidents. Write them down and test them.

**Where this leads.** The exercises stress the boundaries; Chapter 15 replaces one output per slice with one output per prefix.

**Next:** [Chapter 15 — Scan Operations](../ch15_scans/15_complete.ipynb)

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../../part5_layouts/ch13_memory_optimization/13_complete.ipynb)

# Exercises

**Connection.** Builds on the pitfalls and self-check; each exercise targets one failure mode or one boundary case.

1. **Reduction tree.** Draw the sum tree for $N=8$ and count the additions. Compare with $N-1$ serial additions.
2. **Shape rule.** For a $(4,5,6)$ tensor, list the output shape of a sum over axis $0$, axis $1$, axis $2$, axes $(0,1)$, and all axes, with and without keepdim.
3. **Axis swap.** Reduce the $2\times3$ matrix along axis $0$ and axis $1$; then transpose it and repeat. Which results are equal, and why?
4. **Masked max.** Change the masked-mean kernel into a masked-max kernel. Which `other` value is correct, and what should an all-masked row return?
5. **Count denominator.** Break the masked mean by dividing by `BLOCK_N`, run it on a row of length $3$ in a block of width $8$, and compute the scale factor.
6. **Variance cancellation.** Run the one-pass and two-pass formulas on $x=10^5+(1,2,3,4)$ in `float32`. Report both errors and explain the difference.
7. **Sample versus population.** Recompute variance and standard deviation with $N-1$ and state which one LayerNorm uses.
8. **Argmax tie.** Construct a row with two maxima. Check PyTorch's first-index rule and Triton's default; then pass the right-tie option if your release supports it.
9. **NaN policy.** Feed `[1.0, nan, 3.0]` to `torch.max`, `torch.argmax`, `tl.max`, and a NaN-filtered `tl.max`. Record the four results.
10. **Fusion traffic.** For an $M\times N$ `float32` matrix, compare the bytes read by separate sum, sum-of-squares, and mean launches with the bytes read by the fused moments kernel.

In [ ]:
def main() -> None:
    print("Running CPU tests ...")
    test_method01()
    test_policy()

    print("\nRunning GPU tests (skipped without CUDA/ROCm) ...")
    for fn in (
        test_reduce_1d,
        test_row_reduce,
        test_moments_axis1,
        test_moments_axis0,
        test_masked_mean,
    ):
        run_if_gpu(fn)


main()